# Kronos Forecasting Server — Colab Host for Argos

Hosts the **Kronos** candlestick foundation model (`shiyu-coder/Kronos`) on a free Colab GPU runtime and exposes it as a FastAPI HTTP endpoint (via ngrok) so the Argos multi-agent trading desk can call it as the upstream numeric-forecasting stage feeding the **Technicals Analyst** and **Risk Manager** agents.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier is fine for Kronos-small/base).
2. Get a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken — you'll be prompted for it below (or add it as a Colab secret named `NGROK_AUTHTOKEN`).
3. Free Colab sessions idle-timeout (~90 min) and hard-cap around 12h, and the ngrok free-tier URL changes every time you restart this notebook — treat this the same way as the Colab/ngrok vLLM fallback pattern: fine for research/dev, re-run to get a fresh URL, and update the endpoint in your agent config each time.


In [ ]:
# 1. Confirm GPU runtime
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# 2. Install Kronos + serving dependencies
import os

REPO_DIR = "/content/Kronos"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/shiyu-coder/Kronos.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — skipping clone. Delete it first if you want a fresh copy.")

req_path = f"{REPO_DIR}/requirements.txt"
assert os.path.isfile(req_path), (
    "requirements.txt not found. This almost always means the `git clone` above failed or "
    "was interrupted partway (network hiccup, or a stale empty /content/Kronos left over from "
    "an earlier run). Check the clone output, run `!rm -rf /content/Kronos`, then re-run this cell."
)

# Colab already ships a CUDA-matched torch (plus numpy/pandas). requirements.txt pins its own
# versions of these, and pip reinstalling torch from scratch is what makes this step slow —
# it's a multi-GB download + rebuild. Strip those three out and install everything else as-is.
skip_pkgs = ("torch", "numpy", "pandas")
with open(req_path) as f:
    lines = [
        l for l in f
        if l.strip() and not l.strip().startswith("#")
        and not l.strip().lower().startswith(skip_pkgs)
    ]
with open("/content/kronos_reqs_trimmed.txt", "w") as f:
    f.writelines(lines)

%pip install -q -r /content/kronos_reqs_trimmed.txt
%pip install -q fastapi uvicorn nest_asyncio yfinance


In [ ]:
# 3. Load tokenizer + model
# Kronos-small = 24.7M params, max_context 512 -> fast, comfortably fits a free T4.
# Swap to "NeoQuasar/Kronos-base" (102.3M) if you want more capacity and still have VRAM headroom.
import sys, torch

REPO_DIR = "/content/Kronos"
if REPO_DIR not in sys.path:
    # `%cd`-ing into the repo doesn't reliably put it on sys.path for Jupyter/Colab kernels,
    # so `from model import ...` can fail to resolve — Colab then offers to `pip install model`,
    # which is a different (wrong) PyPI package. Adding the repo dir explicitly avoids that.
    sys.path.insert(0, REPO_DIR)

from model import Kronos, KronosTokenizer, KronosPredictor

MODEL_NAME = "NeoQuasar/Kronos-small"   # or "NeoQuasar/Kronos-base"
TOKENIZER_NAME = "NeoQuasar/Kronos-Tokenizer-base"
MAX_CONTEXT = 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")

tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_NAME)
model = Kronos.from_pretrained(MODEL_NAME)

predictor = KronosPredictor(model, tokenizer, device=device, max_context=MAX_CONTEXT)
print("Kronos predictor ready.")


In [ ]:
# 4. Request/response schema — aligned with Argos's FuseState numeric-forecast contract
from pydantic import BaseModel, Field
from typing import List, Optional
import pandas as pd

class Candle(BaseModel):
    timestamp: str          # ISO 8601
    open: float
    high: float
    low: float
    close: float
    volume: Optional[float] = 0.0
    amount: Optional[float] = 0.0

class ForecastRequest(BaseModel):
    ticker: str
    candles: List[Candle] = Field(..., description="Historical OHLCV lookback window, oldest first")
    pred_len: int = Field(24, description="Number of future bars to forecast")
    freq: str = Field("D", description="Pandas frequency string for generated future timestamps, e.g. 'D', 'H', '5min'")
    T: float = 1.0
    top_p: float = 0.9
    sample_count: int = 1

class ForecastResponse(BaseModel):
    ticker: str
    pred_len: int
    forecast: List[dict]    # [{timestamp, open, high, low, close, volume, amount}, ...]

def run_forecast(req: ForecastRequest) -> ForecastResponse:
    hist = pd.DataFrame([c.dict() for c in req.candles])
    hist["timestamp"] = pd.to_datetime(hist["timestamp"])

    lookback = len(hist)
    if lookback < 2:
        raise ValueError("Need at least 2 historical candles")
    if lookback > MAX_CONTEXT:
        hist = hist.iloc[-MAX_CONTEXT:]
        lookback = MAX_CONTEXT

    x_df = hist[["open", "high", "low", "close", "volume", "amount"]]
    x_timestamp = hist["timestamp"]
    last_ts = hist["timestamp"].iloc[-1]
    y_timestamp = pd.Series(pd.date_range(start=last_ts, periods=req.pred_len + 1, freq=req.freq)[1:])

    pred_df = predictor.predict(
        df=x_df,
        x_timestamp=x_timestamp,
        y_timestamp=y_timestamp,
        pred_len=req.pred_len,
        T=req.T,
        top_p=req.top_p,
        sample_count=req.sample_count,
    )

    pred_df = pred_df.reset_index().rename(columns={"index": "timestamp"})
    pred_df["timestamp"] = pred_df["timestamp"].astype(str)

    return ForecastResponse(
        ticker=req.ticker,
        pred_len=req.pred_len,
        forecast=pred_df.to_dict(orient="records"),
    )


In [ ]:
# 5. FastAPI app
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Argos Kronos Forecast Service")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "device": device, "max_context": MAX_CONTEXT}

@app.post("/forecast", response_model=ForecastResponse)
def forecast(req: ForecastRequest):
    try:
        return run_forecast(req)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))


In [ ]:
# 6. Expose the FastAPI server publicly — cloudflared quick tunnel (no account/token needed)
#
# pyngrok/ngrok is a known trigger for Colab's automated abuse-detection system: it watches for
# tools commonly used to run persistent background servers off Colab's compute, which sits
# outside Colab's "interactive compute" usage policy, and pyngrok gets flagged for exactly that
# reason (independent of what the server is actually doing). cloudflared's "quick tunnel" does
# the same job — a public HTTPS URL forwarding to localhost:8000 — without that association.

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, time

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

assert public_url, "cloudflared didn't return a URL in time — re-run this cell (and check GPU/network status above)."
print(f"Kronos endpoint is live at: {public_url}")
print(f"  -> health:   {public_url}/health")
print(f"  -> forecast: {public_url}/forecast  (POST)")

# --- Prefer ngrok instead? (has its own known Colab abuse-detection risk, see note above) ---
# from pyngrok import ngrok, conf
# import getpass
# authtoken = getpass.getpass("Enter your ngrok authtoken: ")
# conf.get_default().auth_token = authtoken
# ngrok.kill()
# public_url = ngrok.connect(8000, "http").public_url
# print(f"Kronos endpoint is live at: {public_url}")


In [ ]:
# 7. Run the server (blocks this cell — leave it running; use Runtime > Interrupt to stop)
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()
print("Server started in background thread. Endpoint stays live as long as this notebook/runtime is connected.")


## 8. Smoke test

Pulls a small window of real OHLCV data via `yfinance` and hits your own endpoint end-to-end, confirming the model, FastAPI app, and ngrok tunnel are all wired correctly before pointing Argos at it.

In [ ]:
import time, requests, yfinance as yf

time.sleep(2)  # give uvicorn a moment to bind

df = yf.download("AAPL", period="2y", interval="1d", progress=False)
df = df.reset_index()
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df = df.rename(columns={"Date": "timestamp", "Open": "open", "High": "high",
                         "Low": "low", "Close": "close", "Volume": "volume"})
df["amount"] = 0.0
df["timestamp"] = df["timestamp"].astype(str)

lookback = 400
payload = {
    "ticker": "AAPL",
    "candles": df[["timestamp", "open", "high", "low", "close", "volume", "amount"]].tail(lookback).to_dict(orient="records"),
    "pred_len": 20,
    "freq": "B",
}

r = requests.post(f"{public_url}/forecast", json=payload, timeout=120)
r.raise_for_status()
result = r.json()
print(f"Forecasted {result['pred_len']} bars for {result['ticker']}")
for row in result["forecast"][:5]:
    print(row)


## 9. Wiring this into Argos

- **Endpoint contract**: `POST {public_url}/forecast` with a `ForecastRequest` body (ticker, OHLCV `candles`, `pred_len`, `freq`) → returns a `ForecastResponse` with forecasted OHLCV bars. This is the numeric-forecasting stage that sits upstream of the **Technicals Analyst** and **Risk Manager** nodes in the LangGraph state machine — call it before those nodes run and pass `forecast` into their prompt/tool context.
- **Config**: put `public_url` in your Argos `.env` (e.g. `KRONOS_ENDPOINT_URL`) rather than hardcoding it — a cloudflared quick-tunnel URL, like an ngrok one, changes every time you restart this notebook.
- **Reliability**: this is still a free Colab session behind a quick tunnel, so treat it exactly like the Colab/ngrok vLLM fallback in the RAG assistant — add a timeout + retry (or a stubbed/local fallback) in the client so a dropped Colab session degrades gracefully instead of blocking the whole agent cycle.
- **Keeping the session alive**: Colab free tier disconnects after ~90 min idle, and its automated abuse-detection can also interrupt long-running non-interactive workloads (background servers included) with little warning — if the runtime drops unexpectedly, that's usually why, not a bug in this notebook.
- **Sizing**: `Kronos-small` (24.7M params) is the safe default for a free T4; `Kronos-base` (102.3M) gives better forecasts if VRAM allows — just re-run cell 3 with `MODEL_NAME = "NeoQuasar/Kronos-base"`.
